# ////////////////////////////////////////////////////////////////mycode///////////////////////////////////////////////////////////////



Exercise 3.2: Advanced Pseudo-Random LSB Audio Steganography
============================================================
This implementation uses a sophisticated approach:
1. Pseudo-random sample selection using a seed-based generator
2. Multi-bit embedding (3 LSBs with adaptive selection)
3. Bit scrambling for added security
4. Payload encoding with error detection
"""

In [1]:
#importing libraries
import librosa
import numpy as np
import scipy.io.wavfile as wav
import IPython.display as ipd

# UTILITY FUNCTIONS

In [2]:
#converting text to a bitstream 
def text_to_bitstream(text):
    bitstream = ''
    for char in text:
        #convert each character to its 8bit binary representation
        bitstream += format(ord(char), '08b')
    return bitstream

In [3]:
#converting a bitstream back to text
def bitstream_to_text(bitstream):
    chars = []
    for i in range(0, len(bitstream), 8):
        #taking 8 bits at a time and convert to character
        byte = bitstream[i:i+8]
        if len(byte) == 8:
            chars.append(chr(int(byte, 2)))
    return ''.join(chars)

In [4]:
#scramblling the bits of the bitstream using a random seed
def scramble_bits(bitstream, seed=42):
    np.random.seed(seed)
    scrambled = ''
    for bit in bitstream:
        #XOR each bit with a random bit to scramble
        random_bit = np.random.randint(0, 2)
        scrambled_bit = int(bit) ^ random_bit
        scrambled += str(scrambled_bit)
    return scrambled

In [5]:
#unscramble bits by calling the scramble function again with the same seed
def unscramble_bits(bitstream, seed=42):
    return scramble_bits(bitstream, seed)

In [6]:
#generate random sample positions within the audio data for embedding information
def generate_sample_positions(total_samples, num_positions_needed, seed=12345):
    np.random.seed(seed)
    #skiping first 1000 samples to avoid overflows
    valid_positions = np.arange(1000, total_samples)
    np.random.shuffle(valid_positions)
    return valid_positions[:num_positions_needed]

In [7]:
#compute checksum of the bitstream 
def compute_checksum(bitstream):
    checksum = 0
    for i in range(0, len(bitstream), 8):
        #XOR each byte to calculate checksum
        byte = bitstream[i:i+8]
        if len(byte) == 8:
            checksum ^= int(byte, 2)
    return format(checksum, '08b')

# EMBEDDING FUNCTION

In [8]:

def embed_message_advanced(audio_file, message, output_file, seed=12345, scramble_seed=42):
 
    print("ADVANCED EMBEDDING:")
    
    #loading audio
    audio, sr = librosa.load(audio_file, sr=None)
    print(f"✓ Audio loaded: {audio_file}")
    print(f"  Sample rate: {sr} Hz")
    print(f"  Duration: {len(audio)/sr:.2f} seconds")
    print(f"  Total samples: {len(audio)}")
    
    #preparing message
    message_bits = text_to_bitstream(message)
    print(f"\n✓ Message: '{message}'")
    print(f"  Length: {len(message)} characters")
    print(f"  Binary length: {len(message_bits)} bits")
    
    #adding checksum for integrity verification
    checksum = compute_checksum(message_bits)
    print(f"  Checksum: {checksum}")
    
    #scramble message bits for security
    scrambled_bits = scramble_bits(message_bits, scramble_seed)
    
    #creating payload
    payload_length = format(len(message_bits), '032b')
    full_payload = payload_length + checksum + scrambled_bits
    
    print(f"\n✓ Payload prepared:")
    print(f"  Header: 32 bits (length) + 8 bits (checksum)")
    print(f"  Total payload: {len(full_payload)} bits")
    
    #converting audio to int16
    audio_int = (audio * 32767).astype(np.int16)
    
    #calculating positions needed 
    bits_per_sample = 3
    positions_needed = (len(full_payload) + bits_per_sample - 1) // bits_per_sample
    
    print(f"\n✓ Embedding strategy:")
    print(f"  Bits per sample: {bits_per_sample} LSBs")
    print(f"  Positions needed: {positions_needed}")
    print(f"  Selection: Pseudo-random (seed={seed})")
    
    #check if we have enough samples
    if positions_needed > len(audio_int) - 1000:
        raise ValueError(f"Audio too short! Need {positions_needed} samples")
    
    #generate pseudorandom sample positions
    sample_positions = generate_sample_positions(len(audio_int), positions_needed, seed)
    print(f"  First 10 positions: {sample_positions[:10]}")
    
    #embed payload
    bit_idx = 0
    samples_used = 0
    
    for sample_pos in sample_positions:
        if bit_idx >= len(full_payload):
            break
        
        #extraacting up to 3 bits from payload
        remaining_bits = len(full_payload) - bit_idx
        bits_to_embed = min(3, remaining_bits)
        
        #getingt the bits to embed
        bits = full_payload[bit_idx:bit_idx + bits_to_embed]
        
        #padding with zeros if less than 3 bits
        if len(bits) < 3:
            bits = bits + '0' * (3 - len(bits))
        
        #converting bits to integer value
        bits_value = int(bits, 2)
        
        #clear 3 LSBs and embed new value
        audio_int[sample_pos] = (audio_int[sample_pos] & ~0x7) | bits_value  #~0x7 clears the 3 LSBs (works with NumPy 1 and 2)
        
        bit_idx += bits_to_embed
        samples_used += 1
    
    print(f"\n✓ Embedding complete:")
    print(f"  Samples modified: {samples_used}")
    print(f"  Percentage of audio: {(samples_used/len(audio_int)*100):.4f}%")
    
    #saving
    wav.write(output_file, sr, audio_int)
    print(f"\n✓ Saved to: {output_file}")
    
    return output_file

# EXTRACTION FUNCTION

In [9]:
def extract_message_advanced(embedded_audio_file, seed=12345, scramble_seed=42):

    print("ADVANCED EXTRACTION: Pseudo-Random Multi-Bit LSB Steganography")
    
    #loading audio
    sr, audio_int = wav.read(embedded_audio_file)
    print(f"✓ Audio loaded: {embedded_audio_file}")
    print(f"  Sample rate: {sr} Hz")
    print(f"  Total samples: {len(audio_int)}")
    
    bits_per_sample = 3
    
    #generating same pseudorandom positions
    #firstly i will to extract enough bits to read the length header 
    header_positions_needed = (32 + bits_per_sample - 1) // bits_per_sample
    sample_positions = generate_sample_positions(len(audio_int), len(audio_int) - 1000, seed)
    
    print(f"\n✓ Extraction strategy:")
    print(f"  Bits per sample: {bits_per_sample} LSBs")
    print(f"  Using seed: {seed}")
    
    #extrating all bits at once
    print(f"\n→ Extracting embedded data...")
    extracted_bits = ''
    pos_idx = 0
    
    #firstly extracting enough positions to get header and message
    #need at least 40 bits for header, but extract more initially
    initial_extraction_samples = 200
    
    for i in range(initial_extraction_samples):
        if pos_idx >= len(sample_positions):
            break
            
        sample_pos = sample_positions[pos_idx]
        
        #extracting 3 LSBs from this sample
        extracted_value = audio_int[sample_pos] & 0b111
        extracted_bits += format(extracted_value, '03b')
        
        pos_idx += 1
    
    #parsing the extracted bits
    #first 32 bits = message length
    message_length = int(extracted_bits[:32], 2)
    print(f"  Message length: {message_length} bits")
    
    #next 8 bits = checksum
    checksum_received = extracted_bits[32:40]
    print(f"  Checksum: {checksum_received}")
    
    #calculating how many total bits needed
    total_bits_needed = 40 + message_length
    
    #extracting more bits if needed
    while len(extracted_bits) < total_bits_needed and pos_idx < len(sample_positions):
        sample_pos = sample_positions[pos_idx]
        extracted_value = audio_int[sample_pos] & 0b111
        extracted_bits += format(extracted_value, '03b')
        pos_idx += 1
    
    print(f"→ Total bits extracted: {len(extracted_bits)}")
    
    #extracting scrambled message
    scrambled_message_bits = extracted_bits[40:40 + message_length]
    
    #unscramble
    message_bits = unscramble_bits(scrambled_message_bits, scramble_seed)
    
    #verifyig checksum
    checksum_calculated = compute_checksum(message_bits)
    print(f"\n✓ Checksum verification:")
    print(f"  Received:   {checksum_received}")
    print(f"  Calculated: {checksum_calculated}")
    print(f"  Status: {'✓ VALID' if checksum_received == checksum_calculated else '✗ INVALID'}")
    
    #converting to text
    extracted_message = bitstream_to_text(message_bits)
    
    print(f"\n✓ Extraction complete!")
    print(f"  Message: '{extracted_message}'")
    
    return extracted_message

# MAIN EXECUTION

In [10]:
if __name__ == "__main__":
    INPUT_AUDIO = 'audio_files/Ex3_sound5.wav'
    OUTPUT_AUDIO = 'Ex3_sound5_embedded.wav'
    SECRET_MESSAGE = "It always seems impossible until it's done"
    
    #seeds for reproducibility
    POSITION_SEED = 12345
    SCRAMBLE_SEED = 42
    #embedding
    try:
        embed_message_advanced(
            audio_file=INPUT_AUDIO,
            message=SECRET_MESSAGE,
            output_file=OUTPUT_AUDIO,
            seed=POSITION_SEED,
            scramble_seed=SCRAMBLE_SEED
        )
        
        print("\n Playing embedded audio...")
        display(ipd.Audio(OUTPUT_AUDIO))
        
    except Exception as e:
        print(f"\n Embedding error: {e}")
        import traceback
        traceback.print_exc()
    #extracting
    try:
        extracted = extract_message_advanced(
            embedded_audio_file=OUTPUT_AUDIO,
            seed=POSITION_SEED,
            scramble_seed=SCRAMBLE_SEED
        )
        
        #verification
        print("FINAL VERIFICATION")
        print(f"Original:  '{SECRET_MESSAGE}'")
        print(f"Extracted: '{extracted}'")
        print(f"Match:     {'✓ SUCCESS' if extracted == SECRET_MESSAGE else '✗ FAILED'}")
        
    except Exception as e:
        print(f"\n Extraction error: {e}")
        import traceback
        traceback.print_exc()
        

ADVANCED EMBEDDING:
✓ Audio loaded: audio_files/Ex3_sound5.wav
  Sample rate: 44100 Hz
  Duration: 9.24 seconds
  Total samples: 407280

✓ Message: 'It always seems impossible until it's done'
  Length: 42 characters
  Binary length: 336 bits
  Checksum: 01111011

✓ Payload prepared:
  Header: 32 bits (length) + 8 bits (checksum)
  Total payload: 376 bits

✓ Embedding strategy:
  Bits per sample: 3 LSBs
  Positions needed: 126
  Selection: Pseudo-random (seed=12345)
  First 10 positions: [  2403 111330  32929 349086 376004 300740 281072 177322 124648 321225]

✓ Embedding complete:
  Samples modified: 126
  Percentage of audio: 0.0309%

✓ Saved to: Ex3_sound5_embedded.wav

 Playing embedded audio...


ADVANCED EXTRACTION: Pseudo-Random Multi-Bit LSB Steganography
✓ Audio loaded: Ex3_sound5_embedded.wav
  Sample rate: 44100 Hz
  Total samples: 407280

✓ Extraction strategy:
  Bits per sample: 3 LSBs
  Using seed: 12345

→ Extracting embedded data...
  Message length: 336 bits
  Checksum: 01111011
→ Total bits extracted: 600

✓ Checksum verification:
  Received:   01111011
  Calculated: 01111011
  Status: ✓ VALID

✓ Extraction complete!
  Message: 'It always seems impossible until it's done'
FINAL VERIFICATION
Original:  'It always seems impossible until it's done'
Extracted: 'It always seems impossible until it's done'
Match:     ✓ SUCCESS


# ////////////////////////////////////////////////////////////////mycode///////////////////////////////////////////////////////////////
